# ML Masterclass 02: Linear Models & Optimization

Welcome to Notebook 02. In this module, we dissect the workhorses of statistical learning: **Linear and Logistic Regression**. 

Most engineers see these as simple baselines. Senior architects recognize them as highly constrained, interpretable marvels that power massive-scale recommender systems and ad-tech click predictors where neural networks are simply too slow or too opaque.

---

## 🎓 Socratic Deep Check
Ponder this question before we begin. The answer lies in the mathematics of geometry:

> *"The Normal Equation gives the exact, mathematically perfect answer for Linear Regression instantly ($w = (X^T X)^{-1} X^T y$). Why do we ever bother using iterative, imprecise Gradient Descent for Linear Regression?"*

---

## Table of Contents
1. **Linear Regression:** The Normal Equation vs Gradient Descent.
2. **Logistic Regression:** Why MSE fails for classification, and the derivation of the Sigmoid.
3. **Optimization Landscapes:** Visualizing Batch, Stochastic, and Mini-batch Gradient Descent tracks in 3D.
4. **Regularization (L1 vs L2):** The geometric reality of sparsity.


## 1. Linear Regression: Exact vs Iterative

We want to find weights $W$ that map input $X$ to output $y$. 
Our loss function is Mean Squared Error (MSE), which is a perfectly smooth, convex bowl $L = \frac{1}{N} \sum (y - Xw)^2$.

### The Analytical Solution (The Normal Equation)
By taking the derivative of the MSE with respect to weights, setting it to zero, and solving for $w$, we get the **Normal Equation**:
$$w = (X^T X)^{-1} X^T y$$

### 🎓 DEEP QUESTION ANSWERED
**Q:** *Why do we use Gradient Descent instead of the Normal Equation?*

**A:** Computational Complexity. The bottleneck is explicitly the matrix inversion step: $(X^T X)^{-1}$. 
The time complexity of converting an $n \times n$ matrix is $O(n^3)$ where $n$ is the number of features. If you have 10 features, it's instant. If you have 100,000 features (like an NLP vocabulary or one-hot encoded categorical IDs), $100000^3 = 10^{15}$ operations. Your RAM will explode, and the CPU will take years. Gradient Descent scales linearly $O(n)$ with the number of features.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

# -----------------------------------------------------
# IMPLEMENTATION: Normal Equation vs Gradient Descent
# -----------------------------------------------------

np.random.seed(42)
X = 2 * np.random.rand(100, 1)          # 100 inputs
y = 4 + 3 * X + np.random.randn(100, 1) # y = 4 + 3x + noise

# Add bias placeholder (x0 = 1) to X
X_b = np.c_[np.ones((100, 1)), X]

print("Task: Find weights [Intercept, Slope]. True weights are [4, 3].\n")

# --- 1. NORMAL EQUATION (Instant Math) ---
w_best_normal = np.linalg.inv(X_b.T.dot(X_b)).dot(X_b.T).dot(y)
print(f"Normal Equation Weights: {w_best_normal.flatten()}")

# --- 2. BATCH GRADIENT DESCENT (Iterative) ---
eta = 0.1  # learning rate
n_iterations = 1000
m = 100    # number of instances

w_gd = np.random.randn(2, 1) # Random initialization

for iteration in range(n_iterations):
    # Transpose(X) dot (Predictions - Y) / m is the mathematical derivative of MSE
    gradients = 2/m * X_b.T.dot(X_b.dot(w_gd) - y)
    w_gd = w_gd - eta * gradients

print(f"Gradient Descent Weights:  {w_gd.flatten()}")

## 2. Logistic Regression: The Classification Bridge

Linear regression outputs values from $-\infty$ to $+\infty$. Classification requires probabilities from $0$ to $1$. 

We squash the linear output ($Xw$) through the **Sigmoid (Logit) function**:
$\sigma(z) = \frac{1}{1 + e^{-z}}$

### 🎓 DEEP QUESTION ANSWERED
**Q:** *Why is Mean Squared Error (MSE) a terrible loss function for Logistic Regression/Classification?*

**A:** If you pass the sigmoid output into an MSE loss function, the resulting mathematical landscape is **non-convex**. It creates a bumpy terrain with multiple local minimums. A gradient descent algorithm will likely get stuck in a bad local valley and fail to find the true global minimum. By switching to **Log Loss (Cross-Entropy)**, the math perfectly cancels out the non-linearities of the sigmoid, guaranteeing a smooth, highly convex bowl with one single global minimum.

In [ ]:
# -----------------------------------------------------
# IMPLEMENTATION: Logistic Regression & Log Loss Custom Loop
# -----------------------------------------------------
from sklearn.datasets import make_classification

# Create binary classification dataset
X_cls, y_cls = make_classification(n_samples=500, n_features=2, n_redundant=0, 
                                   n_clusters_per_class=1, random_state=42)
y_cls = y_cls.reshape(-1, 1)
X_cls_b = np.c_[np.ones((500, 1)), X_cls]

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def cross_entropy_loss(y_true, y_pred):
    # Add tiny epsilon to log to prevent log(0) NaN errors
    epsilon = 1e-15
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

weights_log = np.random.randn(3, 1)
learning_rate = 0.5
loss_history = []

for epoch in range(100):
    # 1. Forward pass (Linear sum -> Sigmoid)
    z = X_cls_b.dot(weights_log)
    predictions = sigmoid(z)
    
    # Track loss
    loss = cross_entropy_loss(y_cls, predictions)
    loss_history.append(loss)
    
    # 2. Backward pass (Derivative of Log Loss + Sigmoid miraculously simplifies to exactly this:)
    gradients = 1/500 * X_cls_b.T.dot(predictions - y_cls)
    
    # 3. Update weights
    weights_log -= learning_rate * gradients

plt.figure(figsize=(8,3))
plt.plot(loss_history)
plt.title("Logistic Regression: Log-Loss Convergence (Custom Loop)")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.show()


## 3. Regularization (L1 vs L2 Matrix Geometry)

Unconstrained models overfit. We force them to be simpler by adding a penalty to the loss function.

*   **L2 (Ridge):** $Loss + \lambda \sum w^2$. Shrinks weights toward zero smoothly. Handles multicollinearity well.
*   **L1 (Lasso):** $Loss + \lambda \sum |w|$. Shrinks weights **exactly to zero**, functioning as built-in Feature Selection.

### 🎓 DEEP QUESTION ANSWERED
**Q:** *Why does L1 uniquely drive coefficients EXACTLY to zero, resulting in sparsity?*

**A:** Think geometrically. The loss function is a 3D bowl, and we are looking for the minimum. Regularization is a rigid physical constraint (a boundary) we place around the origin $(0,0)$. 

The L2 constraint ($w_1^2 + w_2^2 < C$) is a perfect **circle**. When the loss bowl expands and hits the circle, it slides along the edge and stops at a tangent point. That tangent is almost never precisely on the $x$ or $y$ axis, so both weights remain non-zero (just small).

The L1 constraint ($|w_1| + |w_2| < C$) is a **diamond** with sharp, pointy corners resting exactly on the axes. When the loss bowl expands and hits the diamond, the geometric probability of it hitting the sharp corner (which lies exactly on $w_2=0$) is massively higher than hitting the flat edge. Thus, L1 forces $w_2$ to exactly $0.0$, destroying the feature.